# Cortés-Toto et al. (2017): simulation atlas

The published designs include linear 4t/n and a nonlinear Beta-density mixture, with Gaussian noise and optional quarterly seasonality. These are source-inspired computational examples, not reproduced empirical results.

In [ ]:
import awesomefigures as af
from matplotlib_inline.backend_inline import set_matplotlib_formats

# Publication styling only; numerical estimates and metrics are unchanged.
af.set_style("elegante", font="DejaVu Sans", use_tex=False,
             palette="okabe-ito", rasterize=False)
set_matplotlib_formats("svg")

from trendestimation._docs_style import apply_documentation_figure_style
apply_documentation_figure_style()


## Four controlled data-generating cases

Inspect how sample size, nonlinear curvature, and seasonal contamination alter what an in-sample smoother sees.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from trendestimation import simulate_cortes_toto,TrendSmoother,select_lambda
fig,axs=plt.subplots(2,2,figsize=(12,7))
for ax,(shape,season) in zip(axs.flat,[(s,z) for s in ["linear","nonlinear"] for z in [False,True]]):
    ds=simulate_cortes_toto(n=80,shape=shape,seasonal=season,noise_sd=.5,seed=17)
    ax.plot(ds.observed,color="0.72",label="observed");ax.plot(ds.trend,color="k",label="latent trend")
    ax.set(title=f"{shape}; quarterly seasonality={season}")
axs[0,0].legend();fig.tight_layout();plt.show()

## Orders and objective criteria

Within each shape, select $\lambda$ by CV, GCV, AICc, or BIC for $d=1,2,3$. A criterion may prefer distinct smoothness as the shape changes.

In [ ]:
data=simulate_cortes_toto(n=60,shape="nonlinear",noise_sd=.5,seed=7)
fig,axs=plt.subplots(1,3,figsize=(13,3.7))
for ax,d in zip(axs,[1,2,3]):
    sm=TrendSmoother(len(data.observed),d)
    ax.plot(data.observed,color=".82");ax.plot(data.trend,"k--",label="truth")
    for criterion in ["cv","gcv","aicc","bic"]:
        choice=select_lambda(sm,data.observed,criterion=criterion)
        ax.plot(sm.fit(data.observed,lambda_=choice.lambda_).trend,label=criterion.upper())
    ax.set(title=f"Penalty order d={d}",xlabel="Time")
axs[0].legend();fig.tight_layout();plt.show()

## Latent recovery errors: simulation only

For each criterion, compare latent trend recovery RMSE. This is an ex-post evaluation made possible by simulation; it cannot be used as an unseen-data forecasting result.

In [ ]:
names=["cv","gcv","aicc","bic"]
rows=[]
for shape in ["linear","nonlinear"]:
    for n in [50,100]:
        ds=simulate_cortes_toto(n=n,shape=shape,seed=123)
        sm=TrendSmoother(n,2)
        rows.append([np.sqrt(np.mean((sm.fit(ds.observed,lambda_=select_lambda(sm,ds.observed,criterion=c).lambda_).trend-ds.trend)**2)) for c in names])
a=np.asarray(rows)
fig,ax=plt.subplots(figsize=(8,4))
for i,c in enumerate(names):ax.plot(range(len(rows)),a[:,i],"-",label=c.upper())
ax.set(xticks=range(len(rows)),xticklabels=["linear 50","linear 100","nonlinear 50","nonlinear 100"],ylabel="Latent RMSE",title="Simulation diagnostic")
ax.legend();plt.show()